In [0]:
# Gold layer: dim_suppliers (Kimball supplier dimension, Type 1 current-state)
# Source: b2b_ecommerce.silver.suppliers - silver is latest-state-only
#         (SCD Type 1 upsert, one row per supplier_id, latest wins - no
#         history).
# Why Type 1 (KPI-driven, requirements.md "Required KPIs"):
#         KPI 13 (Supplier Revenue: revenue by supplier + concentration
#         % from top N) and KPI 14 (Supplier-Product Performance: best
#         supplier-product combinations) are revenue-attribution metrics
#         that need only supplier IDENTITY. The descriptive supplier
#         attributes (name, country, city, rating) are NOT on the
#         suppliers table - a supplier IS a company (company_id FK), so
#         they come from dim_companies, which is already SCD2 and covers
#         any as-of company attribute. As-of pricing input lives in the
#         supplier_products bridge (supplier_price), not in this dim.
#         No KPI formula uses lead_time_days, min_order_value,
#         is_preferred or contacts, and none needs them time-corrected,
#         so SCD2 would be dead weight here.
# Mechanism: AUTO CDC FROM SNAPSHOT into a Delta-backed streaming table
#         - same ingestion-time pattern as dim_customers (silver cannot
#         be read as a change stream - an upsert target has no change
#         data feed - but it IS a complete current-state snapshot). The
#         view is read at pipeline-run time and the run's ingestion
#         time is the snapshot version (collect-free - SDP forbids
#         driver-side collect()).
# MERGE / idempotency: keyed on supplier_id (requirements' MERGE
#         constraint - safe to retry without duplicates). An identical
#         snapshot diffs to zero events, so retries and no-op runs
#         change nothing. A supplier that disappears from silver is
#         deleted here too (snapshot diff semantics).
# Surrogate key: supplier_sk = crc32(supplier_id) - deterministic,
#         always non-negative and SHORT (unsigned 32-bit, max 10
#         digits; user-directed 2026-11-18 crc32 convention). Negative
#         values stay reserved per the Kimball default-member
#         convention (-1 Unknown, -2 Not Applicable, -3 Error/Invalid)
#         - no seed rows inserted. supplier_id is immutable, so the
#         hash is stable across refreshes; assigned at insert and never
#         rewritten (Type 1 - no versions to distinguish, so unlike
#         dim_companies no updated_at is mixed into the hash).
# Columns: supplier_sk + supplier_id + company_id - the KPI/star-schema
#         set. company_id joins the supplier to dim_companies for
#         name/country/rating (KPI 13 grouping and reporting). The
#         remaining silver columns (lead_time_days, contact_name,
#         contact_email, min_order_value, is_preferred, created_at,
#         updated_at) are used by no KPI and stay out of gold
#         (gold timestamp policy excludes created_at/updated_at).
# Quality: no expectations - silver owns row-level validation, and
#         expectations are not allowed on AUTO CDC FROM SNAPSHOT flows
#         anyway (same decision as the other dims).
# Layout: liquid-clustered by supplier_sk (dims cluster by their
#         surrogate key - KPI views and facts join on it).
from pyspark import pipelines as dp
from pyspark.sql import functions as F

SILVER_SUPPLIERS = "b2b_ecommerce.silver.suppliers"


# Silver's complete current-state snapshot, plus the surrogate key.
# A batch temporary view: the engine reads it at each pipeline run
# (ingestion-time snapshot versioning). No driver-side actions anywhere.
# The select doubles as the column trim: lead_time_days, contact_name,
# contact_email, min_order_value, is_preferred, created_at, updated_at
# stay out of gold (no KPI uses them).
@dp.temporary_view(name="suppliers_snapshot")
def suppliers_snapshot():
    return (
        spark.read.table(SILVER_SUPPLIERS)
        .select(
            # deterministic surrogate key - stable across refreshes,
            # unique per immutable supplier_id, non-negative and short
            # (crc32 is unsigned 32-bit: 0..4,294,967,295)
            F.crc32(F.col("supplier_id").cast("string")).alias("supplier_sk"),
            "supplier_id",
            "company_id",
        )
    )


# Delta-backed streaming table: the AUTO CDC target, clustered on the
# surrogate key (dims cluster by their surrogate key).
dp.create_streaming_table(
    name="dim_suppliers",
    comment=(
        "Kimball supplier dimension, SCD Type 1 (current state): one row per "
        "supplier_id, MERGE-loaded from the b2b_ecommerce.silver.suppliers "
        "current-state snapshot via AUTO CDC FROM SNAPSHOT keyed on supplier_id "
        "- identical snapshots diff to zero events, so retries are idempotent; "
        "rows removed from silver are deleted here. Thin by design: a supplier "
        "IS a company (company_id FK) - descriptive attributes (name, country, "
        "rating) resolve via dim_companies (already SCD2); as-of pricing lives "
        "in the supplier_products bridge. Deterministic surrogate key "
        "supplier_sk = crc32(supplier_id) (unsigned 32-bit, non-negative, "
        "short) is the join key carried by gold facts. SCD2 deliberately not "
        "applied (KPI-driven: KPIs 13/14 need supplier identity only); "
        "non-KPI columns excluded per the gold timestamp policy."
    ),
    cluster_by=["supplier_sk"],
)


# The auto-merge load: diff each snapshot against the last committed
# one and apply inserts/updates/deletes in place (Type 1 - no history
# columns).
dp.create_auto_cdc_from_snapshot_flow(
    target="dim_suppliers",
    source="suppliers_snapshot",  # ingestion-time snapshot versioning
    keys=["supplier_id"],
    stored_as_scd_type=1,
)